# Episode 1: Reading files

## 0. Discovery: what is in this workspace?

Imagine your first day as a data engineer on a workspace you have never seen.
Nobody hands you a diagram. The only thing you know is that the company prefixes its catalogs with **`moustache_markt`**.

Everything else (catalogs, schemas, volumes, where the data actually lives) is **discovered by the code below**, not typed in by hand.

We query **`system.information_schema`** instead of running `SHOW ...` commands:
- it covers **every catalog** you have access to, in one place,
- it is a set of ordinary tables, so it can be filtered, joined and aggregated with SQL,
- the result is a DataFrame, so `collect()` turns it into Python lists we can reuse further down the notebook.

Unity Catalog only returns the objects **you have privileges on**. An empty result can mean "does not exist" or "you are not allowed to see it".

### 0.1 Catalogs

Two SQL details worth knowing:
- **`:prefix` is a named parameter marker.** `spark.sql(..., args={...})` substitutes the value safely, so we never glue values into SQL with f-strings (no quoting bugs, no SQL injection).
- **`startswith()` instead of `LIKE 'moustache_markt%'`.** In `LIKE` the underscore `_` means "any single character", so the pattern would also match e.g. `moustacheXmarkt`.

In [1]:
# The ONLY piece of knowledge we start with: the company's catalog prefix.
# Everything else in this notebook is derived from the workspace itself.
CATALOG_PREFIX = "moustache_markt"

# The same filter is reused in every discovery query below.
# The trailing "_" makes sure we match "moustache_markt_dev", but not "moustache_marktplace".
PREFIX_ARGS = {"prefix": CATALOG_PREFIX + "_"}

catalogs_df = spark.sql(
    """
    SELECT catalog_name, comment, catalog_owner, created, created_by
    FROM system.information_schema.catalogs
    WHERE startswith(catalog_name, :prefix)
    ORDER BY catalog_name
    """,
    args=PREFIX_ARGS,
)
display(catalogs_df)

# collect() brings the rows to the driver as a Python list of Row objects.
# Fine for a handful of metadata rows - never do it on a big table.
# Row fields are accessed by name: row.catalog_name.
CATALOGS = [row.catalog_name for row in catalogs_df.collect()]

# Fail fast: a typo in the prefix or missing permissions should stop the notebook HERE,
# not show up as mysteriously empty results three cells later.
assert CATALOGS, f"No catalogs starting with '{CATALOG_PREFIX}_' - typo in the prefix or no USE CATALOG privilege?"
print(f"Found {len(CATALOGS)} catalog(s): {CATALOGS}")

,catalog_name,comment,catalog_owner,created,created_by
0,moustache_markt_dev,"Data platform, environment: dev",moustache.data.yt.dev@gmail.com,2026-10-12 14:22:48.914,moustache.data.yt.dev@gmail.com
1,moustache_markt_prod,"Data platform, environment: prod",moustache.data.yt.dev@gmail.com,2026-10-12 14:22:54.023,moustache.data.yt.dev@gmail.com
2,moustache_markt_sources,Simulated outside world. Written only by the simulator.,moustache.data.yt.dev@gmail.com,2026-10-12 14:22:41.856,moustache.data.yt.dev@gmail.com


Found 3 catalog(s): ['moustache_markt_dev', 'moustache_markt_prod', 'moustache_markt_sources']


### 0.2 Schemas (and are there any tables in them?)

For every schema we also count its tables with a `LEFT JOIN` to `information_schema.tables`.
A plain `JOIN` would drop schemas that have **no** tables, and those are exactly the ones we want to spot on a fresh workspace.

- `information_schema` is skipped: Unity Catalog creates it automatically in every catalog.
- `default` is also created automatically with every catalog. It is kept, so you can see it is empty.

In [2]:
from pyspark.sql import functions as F

schemas_df = spark.sql(
    """
    SELECT s.catalog_name,
           s.schema_name,
           count(t.table_name) AS tables,   -- count(column) skips NULLs -> 0 for schemas without tables
           s.comment
    FROM system.information_schema.schemata AS s
    LEFT JOIN system.information_schema.tables AS t
           ON t.table_catalog = s.catalog_name
          AND t.table_schema  = s.schema_name
    WHERE startswith(s.catalog_name, :prefix)
      AND s.schema_name <> 'information_schema'
    GROUP BY ALL                             -- group by every non-aggregated column in SELECT
    ORDER BY s.catalog_name, s.schema_name
    """,
    args=PREFIX_ARGS,
)
display(schemas_df)

# One row per catalog: how many schemas, which ones and how many tables in total.
# array_sort makes the list order stable, so catalogs with the same structure look identical
# (collect_list on its own returns elements in whatever order the rows happened to arrive).
display(
    schemas_df.groupBy("catalog_name")
    .agg(
        F.count("*").alias("schemas"),
        F.sum("tables").alias("tables"),
        F.array_sort(F.collect_list("schema_name")).alias("schema_names"),
    )
    .orderBy("catalog_name")
)

SCHEMAS = [(row.catalog_name, row.schema_name) for row in schemas_df.collect()]
TOTAL_TABLES = sum(row.tables for row in schemas_df.collect())
print(f"{len(SCHEMAS)} schema(s), {TOTAL_TABLES} table(s) in total")
if TOTAL_TABLES == 0:
    print("No tables anywhere - if there is any data, it must be sitting in volumes as files.")

,catalog_name,schema_name,tables,comment
0,moustache_markt_dev,bronze_crm,0,"Bronze, source-aligned: crm"
1,moustache_markt_dev,bronze_ecom,0,"Bronze, source-aligned: ecom"
2,moustache_markt_dev,bronze_erp,0,"Bronze, source-aligned: erp"
3,moustache_markt_dev,bronze_pos,0,"Bronze, source-aligned: pos"
4,moustache_markt_dev,default,0,Default schema (auto-created)
5,moustache_markt_dev,gold,0,Gold layer
6,moustache_markt_dev,ops,0,"Operational metadata: checkpoints, DQ results, run logs"
7,moustache_markt_dev,silver,0,Silver layer
8,moustache_markt_prod,bronze_crm,0,"Bronze, source-aligned: crm"
9,moustache_markt_prod,bronze_ecom,0,"Bronze, source-aligned: ecom"


,catalog_name,schemas,tables,schema_names
0,moustache_markt_dev,8,0,"[bronze_crm, bronze_ecom, bronze_erp, bronze_pos, default, gold, ops, silver]"
1,moustache_markt_prod,8,0,"[bronze_crm, bronze_ecom, bronze_erp, bronze_pos, default, gold, ops, silver]"
2,moustache_markt_sources,2,0,"[default, landing]"


18 schema(s), 0 table(s) in total
No tables anywhere - if there is any data, it must be sitting in volumes as files.


### 0.3 Volumes: where are the files?

A **volume** is a Unity Catalog object for **files** (as opposed to tables).
Every volume is mounted under a fixed path, so plain Python (`os`, `open`) can read it like a local disk:

```
/Volumes/<catalog>/<schema>/<volume>/...
```

For each volume we build that path and peek at its **top-level entries only** (no recursion yet, that comes in the next step).
This tells us which volumes actually hold data and how each one is organised.

In [3]:
import os

volumes_df = spark.sql(
    """
    SELECT volume_catalog, volume_schema, volume_name, volume_type, comment
    FROM system.information_schema.volumes
    WHERE startswith(volume_catalog, :prefix)
    ORDER BY volume_catalog, volume_schema, volume_name
    """,
    args=PREFIX_ARGS,
)
display(volumes_df)

# One dict per volume: its full name, its path on the /Volumes mount and its top-level entries.
VOLUMES = []
for row in volumes_df.collect():
    path = f"/Volumes/{row.volume_catalog}/{row.volume_schema}/{row.volume_name}"
    # os.listdir = only the direct children (files and folders), not the whole tree.
    # sorted() because the file system returns entries in no particular order.
    entries = sorted(os.listdir(path))
    VOLUMES.append({
        "full_name": f"{row.volume_catalog}.{row.volume_schema}.{row.volume_name}",
        "path": path,
        "entries": entries,
    })

# Volumes that contain anything at all - the candidates for the rest of this episode.
DATA_VOLUMES = [v for v in VOLUMES if v["entries"]]

# A quick text tree: one line per volume, empty ones clearly marked.
for v in VOLUMES:
    content = ", ".join(v["entries"]) if v["entries"] else "(empty)"
    print(f"{v['full_name']:<45} {content}")

print(f"\n{len(DATA_VOLUMES)} of {len(VOLUMES)} volume(s) contain data:")
for v in DATA_VOLUMES:
    print(f"  {v['path']}")

,volume_catalog,volume_schema,volume_name,volume_type,comment
0,moustache_markt_dev,ops,checkpoints,MANAGED,Auto Loader checkpoints and schema locations
1,moustache_markt_prod,ops,checkpoints,MANAGED,Auto Loader checkpoints and schema locations
2,moustache_markt_sources,landing,crm,MANAGED,Landing zone for source system: crm
3,moustache_markt_sources,landing,ecom,MANAGED,Landing zone for source system: ecom
4,moustache_markt_sources,landing,erp,MANAGED,Landing zone for source system: erp
5,moustache_markt_sources,landing,pos,MANAGED,Landing zone for source system: pos


FileNotFoundError: [Errno 2] No such file or directory: '/Volumes/moustache_markt_dev/ops/checkpoints'

In [0]:
dbutils.fs.ls('/Volumes/moustache_markt_sources/landing/erp')

In [0]:
# variables

FORMATS = {"erp": "parquet", "crm": "jsonl", "ecom": "json_array", "pos": "csv"}

In [0]:
# 1. Pobiera listę volume'ów z information_schema.
#   2. Listuje subfoldery pierwszego poziomu w każdym volume.
#   3. Skanuje równolegle każdy subfolder i zbiera unikalne rozszerzenia.

#   Wszystko działa przez os.scandir, więc nadaje się na serverless.

import os
from collections import defaultdict, Counter
from concurrent.futures import ThreadPoolExecutor

CATALOG = "moustache_markt_sources"

def pathlib_ext(name):
    return "." + name.split(".")[-1] if "." in name else "(no extension)"

def ls(path):
    with os.scandir(path) as it:
        return [(e.path, e.name, e.is_dir()) for e in it]

# 1. Lista volume'ów w katalogu
volumes = spark.sql(f"""
    SELECT volume_schema, volume_name
    FROM {CATALOG}.information_schema.volumes
    ORDER BY volume_schema, volume_name
""")
display(volumes)

samples = {}   # (volume, folder, ext) -> ścieżka pierwszego pliku (alfabetycznie)

# 2. Subfoldery pierwszego poziomu w każdym volume (np. pos/receipt_line)
stats = defaultdict(Counter)   # (volume, folder) -> Counter rozszerzeń
tasks = []                     # ((volume, folder), ścieżka)
for v in volumes.collect():
    root = f"/Volumes/{CATALOG}/{v.volume_schema}/{v.volume_name}"
    for path, name, is_dir in ls(root):
        if is_dir:
            key = (v.volume_name, name)
            stats[key]  # puste foldery też mają się pokazać w wyniku
            tasks.append((key, path))
        else:
            stats[(v.volume_name, "(root)")][pathlib_ext(name)] += 1

# 3. Rekurencyjny skan każdego subfolderu, równolegle, poziom po poziomie
def scan(tasks, workers=32):
    level = tasks
    with ThreadPoolExecutor(max_workers=workers) as pool:
        while level:
            next_level = []
            results = pool.map(ls, [path for _, path in level])
            for (key, _), entries in zip(level, results):
                for path, name, is_dir in entries:
                    if is_dir:
                        next_level.append((key, path))
                    else:
                        stats[key][pathlib_ext(name)] += 1
            level = next_level

scan(tasks)

# Wynik: unikalne rozszerzenia per volume/folder
rows = [
    (vol, folder, sorted(c), dict(c), sum(c.values()))
    for (vol, folder), c in sorted(stats.items())
]
result = spark.createDataFrame(
    rows,
    "volume string, folder string, extensions array<string>, "
    "ext_counts map<string,bigint>, file_count bigint",
)
display(result)

#   Wynik wygląda mniej więcej tak:

#   ┌────────┬──────────────┬───────────────────┬─────────────────────────────┬────────────┐
#   │ volume │    folder    │    extensions     │         ext_counts          │ file_count │
#   ├────────┼──────────────┼───────────────────┼─────────────────────────────┼────────────┤
#   │ erp    │ category     │ [.csv]            │ {.csv → 12}                 │ 12         │
#   ├────────┼──────────────┼───────────────────┼─────────────────────────────┼────────────┤
#   │ pos    │ receipt_line │ [.json, .parquet] │ {.json → 3, .parquet → 480} │ 483        │
#   └────────┴──────────────┴───────────────────┴─────────────────────────────┴────────────┘

#   Kilka uwag:

#   - Pliki leżące bezpośrednio w volume, a nie w subfolderze, trafiają do wiersza (root).
#   - Puste subfoldery też się pokazują, z pustą listą rozszerzeń. Dzięki temu widać, że folder istnieje.
#   - Jeśli w katalogu są inne schematy niż landing i chcesz brać pod uwagę tylko landing, dopisz do
#     zapytania WHERE volume_schema = 'landing'. Jeśli dwa schematy mają volume o tej samej nazwie, dodaj
#     schemat do klucza, inaczej ich statystyki się zsumują.
#   - Kolumna ext_counts to dodatek. Jeśli potrzebujesz tylko unikalnych rozszerzeń, wystarczy kolumna
#     extensions.

In [0]:
# Tak. Najprościej zapamiętać przykładowy plik już podczas skanu, a potem otworzyć go funkcją podglądu.
#   Przykład zapisuję dla każdej pary folder + rozszerzenie, bo w jednym folderze mogą leżeć np. .json i
#   .parquet.

#   1. Zapamiętywanie przykładowego pliku podczas skanu

#   Dodaj ten fragment przed krokiem 2:



def add_file(key, path, name):
    ext = pathlib_ext(name)
    stats[key][ext] += 1
    skey = (*key, ext)
    if skey not in samples or path < samples[skey]:
        samples[skey] = path

#   Następnie w dwóch miejscach podmień linię zliczającą plik na wywołanie add_file.

# W kroku 2 zamień
# stats[(v.volume_name, "(root)")][pathlib_ext(name)] += 1
# na:
# add_file((v.volume_name, "(root)"), path, name)

# W funkcji scan zamień
# stats[key][pathlib_ext(name)] += 1
# na:
# add_file(key, path, name)

# "Pierwszy" plik oznacza tu pierwszy alfabetycznie. scandir zwraca pliki w przypadkowej kolejności, a
# tak przy każdym uruchomieniu dostaniesz ten sam plik.

# 2. Podgląd zawartości

import gzip

def peek(path, n_bytes=2000, n_rows=5):
    name = os.path.basename(path)
    ext = pathlib_ext(name).lower()
    print(f"\n=== {path}")

    # formaty binarne: czyta je Spark i pokazuje kilka wierszy
    if ext in (".parquet", ".orc", ".avro"):
        df = spark.read.format(ext[1:]).load(path)
        df.printSchema()
        display(df.limit(n_rows))
        return

    # pliki skompresowane gzipem: rozpakowuje w locie
    opener = gzip.open if ext == ".gz" else open
    with opener(path, "rb") as f:
        raw = f.read(n_bytes)
    print(raw.decode("utf-8", errors="replace"))

#   Pliki tekstowe (CSV, JSON, TXT, XML) są pokazywane jako surowy tekst. Celowo nie parsuję ich Sparkiem,
#   bo w landing zone chcesz zobaczyć, co faktycznie jest w pliku: separator, nagłówek, czy JSON jest
#   wielolinijkowy, czy plik ma BOM.

#   3. Użycie

#   Podgląd wszystkiego, po jednym pliku na folder i rozszerzenie:

for (vol, folder, ext), path in sorted(samples.items()):
    peek(path)

# Albo podgląd jednego wybranego folderu:

# for (vol, folder, ext), path in samples.items():
#     if (vol, folder) == ("pos", "receipt_line"):
#         peek(path)

# # Ścieżkę przykładowego pliku możesz też dodać jako kolumnę w tabeli wynikowej z poprzedniego kroku:

# rows = [
#     (vol, folder, sorted(c), dict(c), sum(c.values()),
#     {ext: samples[(vol, folder, ext)] for ext in c})
#     for (vol, folder), c in sorted(stats.items())
# ]
# result = spark.createDataFrame(
#     rows,
#     "volume string, folder string, extensions array<string>, "
#     "ext_counts map<string,bigint>, file_count bigint, sample_files map<string,string>",
# )
# display(result)

# Jeśli folderów jest dużo, pętla po wszystkich wyświetli sporo outputu w jednej komórce. Wtedy lepiej
# podglądać po jednym volume na raz.

In [0]:
import json
import os
import sys

import pandas as pd
from pyspark.sql import functions as F

# Repo = folder nad testing/. Dopisujemy go do sys.path, żeby zadziałało `import uc_bootstrap`.
REPO = os.path.dirname(os.getcwd())
if REPO not in sys.path:
    sys.path.insert(0, REPO)
    
from uc_bootstrap import load_config

ENV = "dev"
config = load_config()
SOURCES = config.sources_catalog       # moustache_markt_sources - pliki od systemów źródłowych
CATALOG = config.env_catalog(ENV)      # moustache_markt_dev     - nasza platforma (bronze, silver, gold)
LANDING = config.landing_path          # /Volumes/moustache_markt_sources/landing

# Pierwszy dzień z danych dziennych (01_daily). Tylko w nich jest type corruption, więc testy
# parsowania mają sens przede wszystkim od tej daty.
DAILY_FROM = "2026-06-01"

# Format pliku per system - potrzebny przy czytaniu nagłówków i schematów (krok 5).
FORMATS = {"erp": "parquet", "crm": "jsonl", "ecom": "json_array", "pos": "csv"}

print(f"sources: {SOURCES}\nenv:     {CATALOG}\nlanding: {LANDING}\nsystems: {config.source_systems}")

In [0]:
# Path - an object-oriented file path; works the same on Windows, Linux and in the Databricks Workspace.
from pathlib import Path
DEFAULT_CONFIG_PATH = Path.cwd() / "config" / "platform.yml"
DEFAULT_CONFIG_PATH

In [0]:
config

In [0]:

CATALOG_NAME = 'moustache_markt'
